#### __libraries__

In [156]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import matplotlib.colors as mcolors

In [157]:
from shared_scripts.bb2_data_load import BB2_load_into_dataframe

In [158]:
from shared_scripts.statistical_analysis import iqr_outlier_removal, permutation_test_median

In [159]:
from shared_scripts.additional_functions import extract_significance_symbols

#### __plt params__

In [161]:
plt.rcParams["legend.frameon"] = False
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['axes.labelsize'] = 19
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16  
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 1000
plt.rcParams['font.family'] = 'Calibri'

#### __func__

In [163]:
def fig7(_7_data, **kwargs):
    
    #fig + grid
    fig, ax= plt.subplots(1, 2, figsize= (12.8, 5), constrained_layout= True)  
    
    #order the dataset (if order defined)
    order = kwargs.get('order', [])
    if order:
        unique_strains = _7_data['strain'].unique()
        if len(order) == len(unique_strains) and set(order) == set(unique_strains):
            _7_data['strain'] = pd.Categorical(_7_data['strain'], categories=order, ordered=True)
            _7_data = _7_data.sort_values('strain').reset_index(drop=True)
        else:
            raise ValueError(f'provided strains do not match the strains in the dataset')
    
    #alpha
    box_position = 1.5 
    points_bottom, points_top= 0.25, 0.75 
    colors= plt.cm.tab10(np.linspace(0, 1, 10))
    for i, strain in enumerate(_7_data.strain.unique()):
        dark= '#00527C' if strain == 'WT' else colors[i]
        light= '#DFE9F5' if strain == 'WT' else np.array(mcolors.to_rgb(colors[i])) * 0.3 + 0.7
        data= _7_data.loc[_7_data.strain== strain]
        ax[0].boxplot(data.alpha, 
                      positions=[box_position], 
                      widths=0.5,
                      showfliers=False,  
                      patch_artist= True,
                      boxprops=dict(facecolor=light, edgecolor=dark, linewidth=1.5),
                      whiskerprops=dict(color=dark, linewidth=1.5),
                      capprops=dict(color=dark, linewidth=1.5),
                      medianprops=dict(color=dark, linewidth=2))
        symbols_alpha= kwargs.get('significance_symbols_alpha', None)
        if symbols_alpha:
            significance_symbol_position_y= np.max(iqr_outlier_removal(data.alpha))
            ax[0].annotate(symbols_alpha.get(strain, ''), (box_position, significance_symbol_position_y), fontweight= 'bold', textcoords= 'offset points', xytext= (0, 10), ha= 'center')
        x_points = np.random.uniform(points_bottom, points_top, len(data.alpha))
        ax[0].scatter(x_points,
                      data.alpha,
                      alpha=0.66,
                      color=light,
                      edgecolors="white",
                      linewidths=0.2,
                      s=12,
                      zorder=3)
        
        ax[1].boxplot(data.K, 
                      positions=[box_position], 
                      widths=0.5,
                      showfliers=False,  
                      patch_artist= True,
                      boxprops=dict(facecolor=light, edgecolor=dark, linewidth=1.5),
                      whiskerprops=dict(color=dark, linewidth=1.5),
                      capprops=dict(color=dark, linewidth=1.5),
                      medianprops=dict(color=dark, linewidth=2))
        symbols_k= kwargs.get('significance_symbols_k', None)
        if symbols_k:
            significance_symbol_position_y= np.max(iqr_outlier_removal(data.K))
            ax[1].annotate(symbols_k.get(strain, ''), (box_position, significance_symbol_position_y), fontweight= 'bold', textcoords= 'offset points', xytext= (0, 10), ha= 'center')
        ax[1].scatter(x_points,
                      data.K,
                      alpha=0.66,
                      color=light,
                      edgecolors="white",
                      linewidths=0.2,
                      s=12,
                      zorder=3)
        box_position += 2
        points_bottom += 2
        points_top += 2
    ax[0].set_ylabel('alpha', weight= 'bold')
    ax[0].set_ylim(bottom=-0.55, top= 2.55)
    ax[0].axhline(_7_data.loc[_7_data.strain=='WT', 'alpha'].median(), ls= '--', lw= .66, color="#00527C", alpha= .66, zorder= 5)
    ax[0].set_xlim(0, len(_7_data.strain.unique())*2)
    ax[0].set_xticks(np.arange(1, len(_7_data.strain.unique())*2, 2))
    strain_names= [name if name == 'WT' else f'$\\it{{{name}}}$' for name in _7_data.strain.unique()]
    ax[0].set_xticklabels(strain_names, rotation= 45)
    ax[0].set_yticks([-0.5, 0.5, 1.5, 2.5])
    ax[1].set_ylabel('K', weight= 'bold')
    ax[1].set_ylim(bottom=-2, top= 92)
    ax[1].axhline(_7_data.loc[_7_data.strain=='WT', 'K'].median(), ls= '--', lw= .66, color="#00527C", alpha= .66, zorder= 5)
    ax[1].set_xlim(0, len(_7_data.strain.unique())*2)
    ax[1].set_xticks(np.arange(1, len(_7_data.strain.unique())*2, 2))
    ax[1].set_xticklabels(strain_names, rotation= 45)
    ax[1].set_yticks([0, 30, 60, 90])
    
    ##export
    if kwargs.get('export', False):
        from pathlib import Path
        if not kwargs.get('path'):
            raise ValueError("export path must be provided.")
        output_path = Path(kwargs.get('path'))
        if not output_path.parent.exists():
            raise FileNotFoundError(f"directory does not exist: {output_path}")
        try:
            plt.savefig(output_path, dpi= 1000)
        except Exception as ex:
            raise OSError(f"Failed to export plot to '{kwargs.get('path')}': {ex}")

#### __data load__

In [165]:
_7B_act= BB2_load_into_dataframe('Figure_7B_actin_associated_mutants.csv')

In [166]:
_7C_tub= BB2_load_into_dataframe('Figure_7C_tubulin_associated_mutants.csv')

#### __permutation test__

In [168]:
# permutation_test_median(_7B_act, category_column= 'strain', reference_group= 'WT', tested_parameter= 'alpha', n_resamples= 100000, export= True, path= r"...\Fig_7B_act_alpha_permutation_test.csv")

In [169]:
# permutation_test_median(_7B_act, category_column= 'strain',  reference_group= 'WT', tested_parameter= 'K', n_resamples= 100000, export= True, path= r"...\Fig_7B_act_K_permutation_test.csv")

In [170]:
# permutation_test_median(_7C_tub, category_column= 'strain',  reference_group= 'WT', tested_parameter= 'alpha', n_resamples= 100000, export= True, path= r"...\Fig_7C_tub_alpha_permutation_test.csv")

In [171]:
# permutation_test_median(_7C_tub, category_column= 'strain', reference_group= 'WT', tested_parameter= 'K', n_resamples= 100000, export= True, path= r"...\Fig_7C_tub_K_permutation_test.csv")

#### __significance symbols__

In [174]:
#act mutants
significance_alpha_act= extract_significance_symbols(path= r"...\Fig_7B_act_alpha_permutation_test.csv")
significance_k_act= extract_significance_symbols(path= r"...\Fig_7B_act_K_permutation_test.csv")

#tub mutants
significance_alpha_tub= extract_significance_symbols(path= r"...\Fig_7C_tub_alpha_permutation_test.csv")
significance_k_tub= extract_significance_symbols(path= r"...\Fig_7C_tub_K_permutation_test.csv")

#### __visuals__

In [176]:
fig7(_7B_act,
     order= ['WT', 'tpm1', 'tpm2', 'myo4', 'she3'],
     significance_symbols_alpha= significance_alpha_act,
     significance_symbols_k= significance_k_act,
     export= False,
     path= r"...\Fig7B.png")

In [177]:
fig7(_7C_tub,
     order= ['WT', 'ase1', 'bim1', 'num1', 'cin8', 'kip1', 'kip3'],
     significance_symbols_alpha= significance_alpha_tub,
     significance_symbols_k= significance_k_tub,
     export= False,
     path= r"...\Fig7C.png")